# JSON 파일 로더

JSON(JavaScript Object Notation)은 구조화된 데이터를 저장하는 가장 일반적인 형식입니다.

이 노트북에서는 `JSONLoader`를 사용하여 JSON 파일에서 특정 필드를 추출하고 Document로 변환하는 방법을 학습합니다.

**학습 내용**:
- JSON 구조 분석
- jq_schema로 데이터 경로 지정
- content_key로 본문 필드 선택
- metadata_func로 메타데이터 추가
- 중첩 JSON 처리

# 09. JSON 파일 로더

## 학습 목표
- `JSONLoader`의 `jq_schema`로 JSON 경로를 지정해 원하는 필드를 추출해요
- `content_key`로 `page_content`에 사용할 필드를 선택해요
- `metadata_func`로 JSON의 다른 필드를 메타데이터로 추가해요

## 사전 지식
- 00-Document-Loader 노트북 완료
- JSON 데이터 구조 기본 이해
- `pip install jq` 설치

---

> **RAG 파이프라인 위치**: Load 단계 — API 응답이나 데이터베이스에서 내보낸 JSON을 검색 가능한 Document로 변환해요.

In [1]:
from dotenv import load_dotenv
import json
from pathlib import Path

# 아래에 코드를 작성하세요


## 1. JSON 구조 확인

먼저 JSON 파일의 구조를 확인합니다.

In [2]:
# JSON 파일 읽기

# 아래에 코드를 작성하세요
FILE_PATH = "./data/solar-system.json"

## 2. JSONLoader 기본 사용

`JSONLoader`는 JSON 파일에서 특정 필드를 추출하여 Document로 변환합니다.

**핵심 파라미터**:
- `jq_schema`: JSON 경로를 지정하는 jq 문법 (필수)
- `content_key`: page_content로 사용할 필드명
- `metadata_func`: 메타데이터를 커스터마이즈하는 함수

In [10]:
# ============================================================
# TODO: JSONLoader로 JSON 파일에서 planets 배열의 description 필드를 추출해보세요
# 힌트: JSONLoader(file_path, jq_schema=".planets[]", content_key="description")
# 예상 결과: planets 배열의 각 요소가 별도 Document로 생성됩니다
# ============================================================

from langchain_community.document_loaders import JSONLoader

# 1단계: JSONLoader 생성
# - jq_schema: JSON 경로를 jq 문법으로 지정 (".planets[]" = planets 배열의 각 요소)
# - content_key: 이 필드 값을 page_content로 사용

data = json.loads(Path(FILE_PATH).read_text())
print(f' ==> [Line 13]: \033[38;2;95;119;151m[data]\033[0m({type(data).__name__}) = \033[38;2;85;46;191m{data}\033[0m')

loader = JSONLoader(file_path=FILE_PATH, jq_schema=".planets[]", content_key="description")

docs = loader.load()

docs[0]

 ==> [Line 13]: [data](dict) = {'planets': [{'name': 'Mercury', 'type': 'Terrestrial', 'distance_from_sun': '57.9 million km', 'orbital_period_days': 88, 'description': 'Mercury is the smallest planet in our solar system and the one closest to the Sun. Its surface is heavily cratered and lacks a substantial atmosphere.'}, {'name': 'Venus', 'type': 'Terrestrial', 'distance_from_sun': '108.2 million km', 'orbital_period_days': 225, 'description': 'Venus has a dense atmosphere rich in carbon dioxide, creating a runaway greenhouse effect that makes it the hottest planet.'}, {'name': 'Earth', 'type': 'Terrestrial', 'distance_from_sun': '149.6 million km', 'orbital_period_days': 365, 'description': 'Earth is the only known planet to support life, with abundant liquid water and a balanced atmosphere of nitrogen and oxygen.'}, {'name': 'Jupiter', 'type': 'Gas Giant', 'distance_from_sun': '778.5 million km', 'orbital_period_days': 4333, 'description': 'Jupiter is the largest planet and is known

Document(metadata={'source': '/Users/elixir/dev/lecture/hk-toss/09_langchain/06_RAG/6-1_DocumentLoaders/data/solar-system.json', 'seq_num': 1}, page_content='Mercury is the smallest planet in our solar system and the one closest to the Sun. Its surface is heavily cratered and lacks a substantial atmosphere.')

## 3. metadata_func로 메타데이터 추가

JSON의 다른 필드를 메타데이터로 포함시킬 수 있습니다.

`metadata_func`는 각 레코드를 받아서 메타데이터를 커스터마이즈합니다.

In [14]:
# ============================================================
# TODO: metadata_func을 정의해서 JSON의 name, type 필드를 메타데이터에 추가해보세요
# 힌트: def metadata_func(record, metadata): metadata["name"] = record.get("name"); return metadata
# 예상 결과: 각 Document의 메타데이터에 name, type, distance_from_sun 키가 포함됩니다
# ============================================================

# 1단계: metadata_func 정의
# - record: JSON 배열의 각 요소 (딕셔너리)
# - metadata: 기존 메타데이터 딕셔너리 (source, seq_num 등이 이미 포함됨)
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["name"] = record.get("name", "Unknown")
    metadata["type"] = record.get("type", "Unknown")
    metadata["distance_from_sun"] = record.get("distance_from_sun", "N/A")
    return metadata

# 2단계: metadata_func을 JSONLoader에 전달

loader = JSONLoader(file_path=FILE_PATH, jq_schema=".planets[]", content_key="description", metadata_func=metadata_func)

docs = loader.load()
docs[0].metadata
print(f' ==> [Line 21]: \033[38;2;172;132;109m[docs[0].metadata]\033[0m({type(docs[0].metadata).__name__}) = \033[38;2;71;155;20m{docs[0].metadata}\033[0m')
docs[0].page_content
print(f' ==> [Line 23]: \033[38;2;74;125;211m[docs[0].page_content]\033[0m({type(docs[0].page_content).__name__}) = \033[38;2;204;172;218m{docs[0].page_content}\033[0m')


 ==> [Line 21]: [docs[0].metadata](dict) = {'source': '/Users/elixir/dev/lecture/hk-toss/09_langchain/06_RAG/6-1_DocumentLoaders/data/solar-system.json', 'seq_num': 1, 'name': 'Mercury', 'type': 'Terrestrial', 'distance_from_sun': '57.9 million km'}
 ==> [Line 23]: [docs[0].page_content](str) = Mercury is the smallest planet in our solar system and the one closest to the Sun. Its surface is heavily cratered and lacks a substantial atmosphere.


## 4. 다양한 jq_schema 패턴

jq_schema는 JSON 경로를 유연하게 지정할 수 있습니다.

### jq_schema 예시

| jq_schema | 설명 | 예시 JSON |
|-----------|------|-----------|
| `.` | 전체 JSON | `{"key": "value"}` |
| `.items[]` | items 배열의 각 요소 | `{"items": [1, 2, 3]}` |
| `.data.records[]` | 중첩된 배열 | `{"data": {"records": [...]}}` |
| `.[].content` | 최상위 배열의 content 필드 | `[{"content": "text"}]` |

## 핵심 정리 및 마무리

### jq_schema 패턴 정리

| 패턴 | 설명 | 결과 |
|------|------|------|
| `.` | 전체 JSON | 1개 Document |
| `.items[]` | items 배열의 각 요소 | N개 Document |
| `.data.records[]` | 중첩 배열 | N개 Document |
| `.[].content` | 최상위 배열의 content 필드 | N개 Document |

### 실무 팁
> `jq_schema`가 익숙하지 않으면 먼저 [jq playground](https://jqplay.org/)에서 테스트해 보세요. 잘못된 경로를 지정하면 빈 Document가 생성될 수 있어요.

---

## 다음 예고

- **10-Arxiv-Loader**: 학술 논문을 자동으로 검색하고 로딩
- **11-Directory-Loader**: 폴더 전체를 glob 패턴으로 일괄 처리